# GARCH Baselines vs SFAG

Two classical baselines:
- **GARCH(1,1) — Normal innovations** : lower-bound (clustering only)
- **GJR-GARCH(1,1,1) — Student-t innovations** : strong baseline (clustering + asymmetry + heavy tails)

Both are evaluated with the **exact same `AlignmentModule`** SFAG uses, so the SFAG-gap numbers are directly comparable to the GAN's.

## 0. Setup

In [1]:
import sys, os
sys.path.append('/home/jovyan/SyntheticGenerators/GARCH')

import numpy as np
import torch
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

from data     import load_single_ticker, make_windows
from models   import fit_vanilla_garch, fit_gjr_garch_t, summary
from simulate import simulate_paths
from evaluate import evaluate, print_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

ModuleNotFoundError: No module named 'arch'

## 1. Configuration

In [ ]:
# ── Data ──
DATA_PATH = '/home/jovyan/SyntheticGenerators/StylizedFactsAlignmentGAN/ACM.csv'
TICKER    = 'ACM'

# ── Window length — must match SFAG so the gap metric is comparable ──
T = 2520

# ── Simulation ──
N_SIM_PATHS = 100   # synthetic windows per model
SEED        = 42

# ── SFAG-alignment loss weights (must match SFAG to make numbers comparable) ──
LAMBDA1, LAMBDA2, LAMBDA3, LAMBDA4 = 1.0, 2.0, 0.5, 0.2

## 2. Load Returns

In [ ]:
returns = load_single_ticker(DATA_PATH)
print(f'Returns: {len(returns):,}   mean={returns.mean():+.2e}   std={returns.std():.2e}')

# Real windows for evaluation (matches SFAG layout: (N, T, 1))
real_windows = make_windows(returns, T=T)
print(f'Real windows shape: {real_windows.shape}')

## 3. Fit Both Models

In [ ]:
vanilla = fit_vanilla_garch(returns)
gjr_t   = fit_gjr_garch_t(returns)

print(summary(vanilla))
print()
print(summary(gjr_t))

## 4. Simulate Synthetic Paths

In [ ]:
sim_vanilla = simulate_paths(vanilla, n_paths=N_SIM_PATHS, horizon=T, seed=SEED)
sim_gjr_t   = simulate_paths(gjr_t,   n_paths=N_SIM_PATHS, horizon=T, seed=SEED)

print(f'Vanilla GARCH-N    paths: {sim_vanilla.shape}')
print(f'GJR-GARCH-t        paths: {sim_gjr_t.shape}')

## 5. Evaluate (Same Metrics As SFAG)

In [ ]:
metrics_vanilla = evaluate(
    real_windows, sim_vanilla,
    lambda1=LAMBDA1, lambda2=LAMBDA2,
    lambda3=LAMBDA3, lambda4=LAMBDA4,
    device=device,
)
metrics_gjr_t = evaluate(
    real_windows, sim_gjr_t,
    lambda1=LAMBDA1, lambda2=LAMBDA2,
    lambda3=LAMBDA3, lambda4=LAMBDA4,
    device=device,
)

print_report(metrics_vanilla, title=f'{TICKER} — GARCH(1,1) Normal')
print()
print_report(metrics_gjr_t,   title=f'{TICKER} — GJR-GARCH(1,1,1) Student-t')

## 6. Visual Comparison — Real vs Simulated

In [ ]:
real_s    = real_windows[0, :, 0]
fake_van  = sim_vanilla [0, :, 0]
fake_gjr  = sim_gjr_t   [0, :, 0]

fig, axes = plt.subplots(3, 3, figsize=(16, 11))
fig.suptitle(f'{TICKER} — Real vs GARCH baselines', fontsize=14)

for col, (label, series, color) in enumerate([
    ('Real',           real_s,   'steelblue'),
    ('GARCH-N',        fake_van, 'orange'),
    ('GJR-GARCH-t',    fake_gjr, 'tomato'),
]):
    axes[0, col].plot(series, lw=0.5, color=color)
    axes[0, col].set_title(f'{label} — returns')

    axes[1, col].hist(series, bins=80, color=color, edgecolor='white', lw=0.3)
    axes[1, col].set_title(f'{label} — distribution')

    plot_acf(series ** 2, lags=50, ax=axes[2, col], color=color)
    axes[2, col].set_title(f'{label} — ACF² returns')

plt.tight_layout()
plt.savefig('garch_vs_real_acm.png', dpi=150)
plt.show()

## 7. ACF² Overlay

In [ ]:
lags = np.arange(1, len(metrics_vanilla['acf_real']) + 1)

plt.figure(figsize=(10, 4))
plt.plot(lags, metrics_vanilla['acf_real'], 'o-',  lw=1.5, ms=4, color='steelblue', label='Real')
plt.plot(lags, metrics_vanilla['acf_sim'],  's--', lw=1.5, ms=4, color='orange',    label='GARCH-N')
plt.plot(lags, metrics_gjr_t  ['acf_sim'],  '^--', lw=1.5, ms=4, color='tomato',    label='GJR-GARCH-t')
plt.axhline(0, color='grey', lw=0.8, ls='--')
plt.title(f'ACF of squared returns — {TICKER}')
plt.xlabel('Lag'); plt.ylabel('Autocorrelation')
plt.legend()
plt.tight_layout()
plt.savefig('acf_overlay_garch_acm.png', dpi=150)
plt.show()

## 8. Side-by-Side SFAG-Gap Summary

In [ ]:
print(f"{'Model':<25s} {'SFAG gap':>10s} {'ACF\u00b2 MSE':>12s} {'CFVC gap':>10s} {'GPD gap':>10s} {'Lev gap':>10s}")
print('-' * 80)
for name, m in [('GARCH(1,1) Normal', metrics_vanilla),
                ('GJR-GARCH(1,1,1) t', metrics_gjr_t)]:
    gpd_gap = abs(m['gpd_real'] - m['gpd_sim'])
    lev_gap = abs(m['lev_real'] - m['lev_sim'])
    print(f"{name:<25s} {m['sfag_gap']:>10.4f} {m['acf_mse']:>12.6f} "
          f"{m['cfvc_gap']:>10.4f} {gpd_gap:>10.4f} {lev_gap:>10.4f}")